# 02 - Feature Engineering

Este notebook construye todas las features del pipeline:
1. **Elo ratings** (game-by-game, margin-adjusted)
2. **Efficiency metrics** (KenPom-style: Four Factors, off/def efficiency)
3. **Seed features** (numeric seed, historical win rates)
4. **Massey ordinals** (top ranking systems aggregated)
5. **Conference features** (conference strength, power conference flag)
6. **Win/loss record** features
7. **Matchup-level training dataset** (differences entre teams)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add utils to path
sys.path.insert(0, str(Path('.').resolve()))

from utils.elo import EloSystem, build_elo_features
from utils.efficiency import build_efficiency_features
from utils.features import (
    build_seed_features, compute_historical_seed_win_rates,
    aggregate_massey_ordinals, build_conference_features,
    build_record_features, build_team_features,
    build_training_data, build_submission_predictions,
)

DATA_DIR = Path('../data')
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
# Load all data
m_reg_compact = pd.read_csv(DATA_DIR / 'MRegularSeasonCompactResults.csv')
m_reg_detailed = pd.read_csv(DATA_DIR / 'MRegularSeasonDetailedResults.csv')
m_tourney = pd.read_csv(DATA_DIR / 'MNCAATourneyCompactResults.csv')
m_seeds = pd.read_csv(DATA_DIR / 'MNCAATourneySeeds.csv')
m_massey = pd.read_csv(DATA_DIR / 'MMasseyOrdinals.csv')
m_team_conf = pd.read_csv(DATA_DIR / 'MTeamConferences.csv')
m_teams = pd.read_csv(DATA_DIR / 'MTeams.csv')

w_reg_compact = pd.read_csv(DATA_DIR / 'WRegularSeasonCompactResults.csv')
w_reg_detailed = pd.read_csv(DATA_DIR / 'WRegularSeasonDetailedResults.csv')
w_tourney = pd.read_csv(DATA_DIR / 'WNCAATourneyCompactResults.csv')
w_seeds = pd.read_csv(DATA_DIR / 'WNCAATourneySeeds.csv')
w_team_conf = pd.read_csv(DATA_DIR / 'WTeamConferences.csv')

sub_stage1 = pd.read_csv(DATA_DIR / 'SampleSubmissionStage1.csv')
sub_stage2 = pd.read_csv(DATA_DIR / 'SampleSubmissionStage2.csv')

print('Data loaded:')
print(f'  Men reg compact: {m_reg_compact.shape}')
print(f'  Men reg detailed: {m_reg_detailed.shape}')
print(f'  Men tourney: {m_tourney.shape}')
print(f'  Men seeds: {m_seeds.shape}')
print(f'  Massey ordinals: {m_massey.shape}')
print(f'  Women reg compact: {w_reg_compact.shape}')
print(f'  Women reg detailed: {w_reg_detailed.shape}')
print(f'  Women tourney: {w_tourney.shape}')

## 1. Elo Ratings

In [ ]:
%%time
# Build Elo ratings for men's and women's
# Only use regular season for features (tourney is what we predict)
m_elo_df, w_elo_df, m_elo_sys, w_elo_sys = build_elo_features(
    m_reg_compact, m_team_conf,
    w_reg_compact, w_team_conf,
    k=32, home_advantage=100, season_carryover=0.65,
)

print(f"Men's Elo: {m_elo_df.shape}")
print(f"Women's Elo: {w_elo_df.shape}")
print(f"\nMen's Elo stats:")
print(m_elo_df['Elo'].describe())

In [ ]:
# Sanity check: top teams in latest season should match known powerhouses
latest_season = m_elo_df.Season.max()
top_elo = m_elo_df[m_elo_df.Season == latest_season].nlargest(20, 'Elo')
top_elo = top_elo.merge(m_teams[['TeamID', 'TeamName']], on='TeamID')

print(f"Top 20 Men's Elo ratings (Season {latest_season}):")
for i, (_, row) in enumerate(top_elo.iterrows(), 1):
    print(f"  {i:2d}. {row['TeamName']:25s} | Elo: {row['Elo']:.0f}")

# Plot Elo distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(m_elo_df[m_elo_df.Season == latest_season]['Elo'], bins=40, color='steelblue', alpha=0.7)
axes[0].set_title(f"Men's Elo Distribution (Season {latest_season})")
axes[0].set_xlabel('Elo Rating')

# Elo of 1-seeds over time
seed1_elo = m_seeds[m_seeds.Seed.str.contains('01')].merge(m_elo_df, on=['Season', 'TeamID'])
axes[1].scatter(seed1_elo['Season'], seed1_elo['Elo'], alpha=0.5, s=20)
axes[1].plot(seed1_elo.groupby('Season')['Elo'].mean(), color='red', linewidth=2)
axes[1].set_title('#1 Seeds Elo Over Time')
axes[1].set_ylabel('Elo')

plt.tight_layout()
plt.show()

## 2. Efficiency Metrics

In [ ]:
%%time
# Build efficiency features (only from detailed results, 2003+)
m_eff_df = build_efficiency_features(m_reg_detailed, max_day=132, last_n_games=10)
w_eff_df = build_efficiency_features(w_reg_detailed, max_day=132, last_n_games=10)

print(f"Men's efficiency features: {m_eff_df.shape}")
print(f"Women's efficiency features: {w_eff_df.shape}")
print(f"\nColumns: {m_eff_df.columns.tolist()}")
print(f"\nSample (top 5 by NetEff, latest season):")
latest = m_eff_df[m_eff_df.Season == m_eff_df.Season.max()].nlargest(5, 'NetEff')
latest = latest.merge(m_teams[['TeamID', 'TeamName']], on='TeamID')
print(latest[['TeamName', 'OffEff', 'DefEff', 'NetEff', 'eFG', 'WinPct']].to_string(index=False))

## 3. Seed Features

In [ ]:
# Historical seed matchup win rates
m_seed_rates = compute_historical_seed_win_rates(m_tourney, m_seeds)
w_seed_rates = compute_historical_seed_win_rates(w_tourney, w_seeds)

print(f"Men's seed matchups: {len(m_seed_rates)} unique matchups")
print(f"Women's seed matchups: {len(w_seed_rates)} unique matchups")

# Show first-round matchup rates
first_round = m_seed_rates[m_seed_rates.StrongSeed + m_seed_rates.WeakSeed == 17].sort_values('StrongSeed')
print("\nFirst round matchup historical win rates (Men's):")
print(first_round.to_string(index=False))

## 4. Massey Ordinals

In [ ]:
%%time
# Aggregate Massey ordinals (all systems)
m_massey_agg = aggregate_massey_ordinals(m_massey, max_day=133)

print(f"Men's Massey aggregated: {m_massey_agg.shape}")
print(f"\nSample (top 5 by MasseyMean, latest season):")
latest_massey = m_massey_agg[m_massey_agg.Season == m_massey_agg.Season.max()].nsmallest(5, 'MasseyMean')
latest_massey = latest_massey.merge(m_teams[['TeamID', 'TeamName']], on='TeamID')
print(latest_massey[['TeamName', 'MasseyMean', 'MasseyMedian', 'MasseyStd', 'MasseyCount']].to_string(index=False))

## 5. Conference & Record Features

In [ ]:
# Conference features
m_conf_df = build_conference_features(m_team_conf, m_elo_df)
w_conf_df = build_conference_features(w_team_conf, w_elo_df)

# Record features
m_record_df = build_record_features(m_reg_compact, max_day=132)
w_record_df = build_record_features(w_reg_compact, max_day=132)

print(f"Men's conference features: {m_conf_df.shape}")
print(f"Men's record features: {m_record_df.shape}")
print(f"\nConference feature columns: {m_conf_df.columns.tolist()}")
print(f"Record feature columns: {m_record_df.columns.tolist()}")

## 6. Merge All Team Features

In [ ]:
# Build unified team features for men's
m_team_feats = build_team_features(
    elo_df=m_elo_df,
    efficiency_df=m_eff_df,
    seeds_df=m_seeds,
    massey_df=m_massey_agg,
    conf_df=m_conf_df,
    record_df=m_record_df,
)

# Build unified team features for women's
# Note: Women's don't have Massey ordinals in the data
w_massey_agg = pd.DataFrame()  # Empty - no Massey for women
w_team_feats = build_team_features(
    elo_df=w_elo_df,
    efficiency_df=w_eff_df,
    seeds_df=w_seeds,
    massey_df=w_massey_agg,
    conf_df=w_conf_df,
    record_df=w_record_df,
)

print(f"Men's team features: {m_team_feats.shape}")
print(f"Women's team features: {w_team_feats.shape}")
print(f"\nMen's feature columns ({len(m_team_feats.columns)}):")
print(m_team_feats.columns.tolist())
print(f"\nMissing values:")
missing = m_team_feats.isnull().sum()
print(missing[missing > 0])

## 7. Build Training Dataset

In [ ]:
# Define feature columns for matchup differences
# These are the numeric columns from team features
exclude_cols = {'Season', 'TeamID', 'ConfAbbrev'}
m_feature_cols = [
    c for c in m_team_feats.columns
    if c not in exclude_cols and m_team_feats[c].dtype in ['float64', 'int64', 'float32', 'int32']
]
w_feature_cols = [
    c for c in w_team_feats.columns
    if c not in exclude_cols and w_team_feats[c].dtype in ['float64', 'int64', 'float32', 'int32']
]

print(f"Men's feature columns for matchup: {len(m_feature_cols)}")
print(m_feature_cols)
print(f"\nWomen's feature columns for matchup: {len(w_feature_cols)}")
print(w_feature_cols)

In [ ]:
%%time
# Build men's training data from tournament games
m_train = build_training_data(m_tourney, m_team_feats, m_feature_cols)
# Add gender flag
m_train['is_mens'] = 1

print(f"Men's training data: {m_train.shape}")
print(f"Target distribution: {m_train['target'].value_counts().to_dict()}")
print(f"Seasons: {m_train['Season'].min()} - {m_train['Season'].max()}")

In [ ]:
%%time
# Build women's training data
w_train = build_training_data(w_tourney, w_team_feats, w_feature_cols)
w_train['is_mens'] = 0

print(f"Women's training data: {w_train.shape}")
print(f"Target distribution: {w_train['target'].value_counts().to_dict()}")
print(f"Seasons: {w_train['Season'].min()} - {w_train['Season'].max()}")

In [ ]:
# Combine men's and women's training data
# Find common feature columns (women may not have Massey)
common_cols = sorted(set(m_train.columns) & set(w_train.columns))
train_full = pd.concat([m_train[common_cols], w_train[common_cols]], ignore_index=True)

print(f"Combined training data: {train_full.shape}")
print(f"Feature columns: {[c for c in common_cols if c not in ["Season", "TeamID1", "TeamID2", "target", "is_mens"]]}'")
print(f"\nTarget distribution: {train_full['target'].value_counts().to_dict()}")
print(f"Gender distribution: {train_full['is_mens'].value_counts().to_dict()}")
print(f"\nMissing values per column:")
missing = train_full.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Save training data and team features for modeling notebooks
train_full.to_csv(DATA_DIR / 'train_matchups.csv', index=False)
m_team_feats.to_csv(DATA_DIR / 'team_features_mens.csv', index=False)
w_team_feats.to_csv(DATA_DIR / 'team_features_womens.csv', index=False)

print('Saved:')
print(f'  train_matchups.csv: {train_full.shape}')
print(f'  team_features_mens.csv: {m_team_feats.shape}')
print(f'  team_features_womens.csv: {w_team_feats.shape}')

In [ ]:
# Quick correlation analysis of top features
feature_diff_cols = [c for c in train_full.columns if c.endswith('_diff')]

if len(feature_diff_cols) > 0:
    # Correlation with target
    corr_with_target = train_full[feature_diff_cols + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)
    
    print("Feature correlations with target (absolute):")
    print(corr_with_target.head(20).to_string())
    
    fig, ax = plt.subplots(figsize=(12, 8))
    corr_with_target.head(20).plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 20 Features by Correlation with Target')
    ax.set_xlabel('|Correlation|')
    plt.tight_layout()
    plt.show()